<a href="https://colab.research.google.com/github/Oguipereira/AnaliseCorretoresDados/blob/main/Mensura%C3%A7%C3%A3oCorretores.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!git clone https://github.com/Oguipereira/AnaliseCorretoresDados

fatal: destination path 'AnaliseCorretoresDados' already exists and is not an empty directory.


In [ ]:
%cd AnaliseCorretoresDados

/content/AnaliseCorretoresDados


In [ ]:
!pip install pyxlsb openpyxl -q

Upload dos arquivos

In [ ]:
from google.colab import files

print("Selecione a base de carteira (.xlsb)")
arquivos = files.upload()

print("Selecione a planilha de solicitações (.xlsx)")
arquivos = files.upload()

Selecione a base de carteira (.xlsb)


KeyboardInterrupt: 

Bloco de configuração

In [ ]:
ARQUIVO_BASE = "Cópia_AZ_Base_Codigos_Corretores_07_2026_Final.xlsb"

ARQUIVO_SOLICITACOES = "Cópia de cadastroz BeSafe 1708_3617.xlsx"

ABA_BASE = "Az_Cod_Corretores"

COLUNA_ID = "CNPJ_CPF"

COLUNAS_PARA_PUXAR = {
    "SUSEP_FILHO": "SUSEP_FILHO",
    "REGIONAL": "REGIONAL",
    "FILIAL_DS": "FILIAL_DS",
    "FILIAL_CD": "FILIAL_CD",
    "CHAVE": "CHAVE",
    "ASSESSORIA": "ASSESSORIA",
    "NOME_ASSESSORIA": "NOME_ASSESSORIA",
    "AGRUPAMENTO?": "AGRUPAMENTO?",
    "MOTIVO DE ENCERRAMENTO": "MOTIVO DE ENCERRAMENTO",
    "ATENDIMENTO": "ATENDIMENTO",
}

Bloco de funções

In [ ]:
import pandas as pd
import re

def limpar_documento(valor):

    if pd.isna(valor):
        return ""

    numeros = re.sub(r"\D", "", str(valor))

    if len(numeros) <= 11:
        return numeros.zfill(11)

    return numeros.zfill(14)

Bloco de leitura

In [ ]:
df_base = pd.read_excel(
    ARQUIVO_BASE,
    sheet_name=ABA_BASE,
    dtype=str,
    engine="pyxlsb"
)

df_solicitacoes = pd.read_excel(
    ARQUIVO_SOLICITACOES,
    dtype=str
)

print("Base:", df_base.shape)
print("Solicitações:", df_solicitacoes.shape)

Bloco de preparação da carteira

In [ ]:
df_base["_ID_LIMPO"] = (
    df_base["CNPJ_CPF"]
    .apply(limpar_documento)
)

colunas = ["CNPJ_CPF"] + list(COLUNAS_PARA_PUXAR.values())

df_base = df_base[colunas].copy()

df_base["_ID_LIMPO"] = (
    df_base["CNPJ_CPF"]
    .apply(limpar_documento)
)

df_base = df_base.drop_duplicates(
    subset="_ID_LIMPO"
)

Bloco de preparação de solicitações

In [ ]:
df_solicitacoes["_ID_LIMPO"] = (
    df_solicitacoes["CNPJ_CPF"]
    .apply(limpar_documento)
)

Bloco da "PROCV" na carteira

In [ ]:
base_lookup = df_base.drop(
    columns=["CNPJ_CPF"]
)

resultado = df_solicitacoes.merge(
    base_lookup,
    on="_ID_LIMPO",
    how="left",
    indicator=True
)

Bloco de separação de encontrados e não encontrados

In [ ]:
encontrados = (
    resultado[
        resultado["_merge"] == "both"
    ]
    .copy()
)

nao_encontrados = (
    resultado[
        resultado["_merge"] == "left_only"
    ]
    .copy()
)

Bloco de exportação

In [ ]:
with pd.ExcelWriter(
    "Output_consolidado.xlsx",
    engine="openpyxl"
) as writer:

    resultado.to_excel(
        writer,
        sheet_name="Analise_Cadastral",
        index=False
    )

    encontrados.to_excel(
        writer,
        sheet_name="Encontrados",
        index=False
    )

    nao_encontrados.to_excel(
        writer,
        sheet_name="Nao_Encontrados",
        index=False
    )

print("Arquivo gerado!")

Bloco de upload do output

In [ ]:
from google.colab import files

files.download(
    "Output_consolidado.xlsx"
)